In [ ]:
# ==========================================
# Step 1: Install the current official unified SDK (2026).
# ==========================================
print("📦 正在安装最新版 google-genai 库...")
!pip install google-genai -q
print("✅ 安装完成！\n")

# ==========================================
# Step 2: Obtain the underlying Application Default Credentials (ADC) with the necessary permissions.
# ==========================================
print("🔑 准备进行底层授权...")
print("⚠️ 请注意：下方会生成一个蓝色链接。请点击它，并使用带有 300 刀赠金的 Google 账号登录授权。")
print("⚠️ 授权后复制验证码，粘贴回下方的输入框中并回车。")
!gcloud auth application-default login

# ==========================================
# Step 3: Assign the new project as the quota and billing project.
# ==========================================
print("\n💰 正在将计费接口挂载到你的新项目 [api-try-495100]...")
!gcloud auth application-default set-quota-project api-try-495100
print("✅ 配额注入成功！\n")

# ==========================================
# Step 4: Call Gemini 2.5 Flash.
# ==========================================
from google import genai

print("🔄 正在连接到自由的新世界...")

try:
    # Set vertexai=True to route requests through Vertex AI.
    client = genai.Client(
        vertexai=True,
        project="api-try-495100",
        location="us-central1"
    )

    print("🚀 正在使用 gemini-2.5-flash 模型发送请求...")
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="你好，总指挥！这是从全新的无组织专属项目中发出的第一条测试消息。如果你收到了，请用极其热烈的语气恭喜我'金蝉脱壳'彻底成功！"
    )

    print("\n" + "="*40)
    print("🎉🎉🎉 彻底成功！模型回复如下：\n")
    print(response.text)
    print("="*40 + "\n")
    print("💡 下一步：你现在可以放心地把这个初始化逻辑，无缝接入到你的 CSV 批处理循环中了！")

except Exception as e:
    print("\n❌ 报错了，请检查信息：", e)

In [ ]:
# -*- coding: utf-8 -*-

import json
import time
import random
import hashlib
import sqlite3
import re
import requests

from datetime import datetime
from pathlib import Path
import pandas as pd

### 1. Input and output

In [ ]:
from pathlib import Path

INPUT_FILE = Path(r"/content/drive/MyDrive/CES分析/源数据-重分类/公园/海淀公园/海淀公园.csv")

OUT_DIR = Path(r"/content/drive/MyDrive/CES分析/评论-实验")
OUT_DIR.mkdir(parents=True, exist_ok=True)

### 2. API configuration

### 3. Performance and reliability settings

In [ ]:
# =========================
# 3. Performance and reliability settings
# =========================
# Increase this value to reduce API call frequency and avoid HTTP 429 errors.
MIN_INTERVAL_SEC = 3.0 # The previous value was 1.8.
_last_call = 0.0

WRITE_EVERY = 30              # Number of records processed between CSV saves.
TEXT_MAX_CHARS = 900          # Truncate long reviews to improve stability and limit latency and cost.
TIMEOUT = (10, 40)            # Connection and read timeouts, in seconds.
MAX_RETRIES = 5               # Maximum retries per record (previously 4).
MAX_TOTAL_SEC_PER_ITEM = 90   # Maximum total processing time per record.
MIN_CONFIDENCE_FOR_FALLBACK = 0.6 # Consider a lexicon fallback if the model returns confidence below this value.

### 4. CES categories

In [ ]:
# =========================
# 4. CES categories and output columns for evidence and sentiment
# =========================
CES_CLASSES = [
    "Spiritual_and_religious_value",
    "Knowledge_and_education",
    "Inspiration",
    "Aesthetic_values_nature",
    "Aesthetic_values_architecture",
    "Aesthetic_values_landscape",
    "Social_relations",
    "Sense_of_place",
    "Cultural_heritage_values",
    "Recreation_and_visitation"
]

# Create two columns for each category: EV (evidence) and SNT (sentiment).
CES_DATA_COLS = []
for c in CES_CLASSES:
    CES_DATA_COLS.append(f"EV_{c}")
    CES_DATA_COLS.append(f"SNT_{c}")

In [ ]:
import sys
from google.colab import drive

# 1. Mount Google Drive; if it is not mounted, an authorisation dialog appears.
try:
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
    print("✅ Drive 状态: 已挂载")
except Exception as e:
    print(f"❌ Drive 挂载失败: {e}")

# 2. Check that the key variables have been defined.
required_vars = ['INPUT_FILE', 'SYSTEM', 'call_api', 'read_csv_robust']
missing = [v for v in required_vars if v not in globals()]
if missing:
    print(f"⚠️ 警告: 缺少关键定义 {missing}。请确保运行了上方所有 cell_id 为 new_cell_x 的单元格。")
else:
    print(f"🚀 准备就绪，开始处理文件: {INPUT_FILE}")
    # 3. Run the main workflow manually using the code in step 10.
    try:
        # Call the previously defined logic to label records in a loop.
        # If the main workflow has not run yet, run the cell identified by cell_id: new_cell_20 below.
        print("请点击运行下方的 '⑩ 主流程' 单元格来开始打标工作。")
    except NameError:
        print("请先从上到下依次点击每个代码块左侧的播放按钮。")

### 5. Lexicon for prompting and fallback

In [ ]:
# =========================
# 5. Lexicon updated from the latest supplied terms
# =========================
LEXICON = {
    "Sense_of_place": ["氛围","感觉","意境","代入感","沉浸","身临其境","古韵","古风","古意","历史氛围","文化氛围","园林气息","皇家气息","古典气息","气息","地方感","归属感","熟悉感","亲切感","老北京","北京味道","京味","京味文化"],
    "Spiritual_and_religious_value": ["祈福","祈愿","祈祷","许愿","虔诚","上香","祠","寺庙","庙宇","佛","菩萨","道观","神圣","信仰","宗教","香火","拜佛","烧香","供奉","求签","祭祀","祭祠","宁静","安静","清幽","幽静","静谧","安逸","放松","治愈","惬意","舒适","悠闲","松弛"],
    "Knowledge_and_education": ["知识","科普","讲解","讲解员","解说","了解","学习","介绍","说明","展览","展厅","展馆","博物馆","历史介绍","文化讲解","教育","孩子","学生","研学","课堂","长见识"],
    "Inspiration": ["灵感","启发","激发","创作","思考","感悟","感想","触动","诗意","写生","画画","摄影创作"],
    "Aesthetic_values_nature": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","景色","风景","美景","景致","风光","景观","园林美","画面","视觉","色彩","光影","倒影","构图","层次","景观设计","拍照","摄影","出片","拍照好看","适合拍照","打卡拍照","夜景","灯光","日落","日出","夕阳","晨光","树","树木","树林","森林","古树","绿树","绿荫","花","花海","花园","花坛","荷花","樱花","梅花","草","草地","草坪","湖","湖水","湖面","池塘","水面","溪流","荷塘","水景"],
    "Aesthetic_values_architecture": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","建筑","亭","台","楼","阁","殿","宫","院","廊","长廊","回廊","桥","石桥","拱桥","小桥","门","宫门","山门","牌楼","牌坊","碑","石碑","碑刻","寺","庙","观","祠","古建","古建筑"],
    "Aesthetic_values_landscape": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","园林","公园","景区","景点","整体景观","园子","园子里"],
    "Social_relations": ["朋友","家人","亲子","情侣","一起","陪伴","同行","合影","拍合照","约会","聚会","团建","带父母","带孩子"],
    "Cultural_heritage_values": ["历史","古迹","文物","文化","传统","遗产","皇家","宫廷","园林文化","历史感","年代感","历史厚重","文化底蕴","文化气息","文化价值","保护","修复","修缮","保护文物","保护遗址","传承","历史价值","亭","台","楼","阁","殿","宫","院","园","廊","长廊","回廊","桥","门","牌楼","牌坊","碑","石碑","碑刻","题字","匾额","寺","庙","观","祠","遗址","遗迹","城墙","古建","古建筑"],
    "Recreation_and_visitation": ["游玩","逛","逛园","逛逛","游览","参观","看看","散步","溜达","遛弯","打卡","打卡点","网红打卡","旅游","旅行","景点","游客","很多人来","推荐","值得去","值得一看","排队","门票","慢慢逛","随便逛"]
}

# Define terms that should not trigger Spiritual_and_religious_value.
EXCLUDE_FROM_SPIRITUAL = ["老佛爷", "佛香阁"]

def lexicon_hint_labels(text: str):
    t = text or ""
    hits = set()
    for lab, words in LEXICON.items():
        if lab == "Spiritual_and_religious_value":
            # Do not assign Spiritual_and_religious_value if the review contains an exclusion term.
            if any(ex_word in t for ex_word in EXCLUDE_FROM_SPIRITUAL):
                continue
        if any(w in t for w in words):
            hits.add(lab)
    return [c for c in CES_CLASSES if c in hits]

def evidence_from_text(label: str, text: str, max_len: int = 80) -> str:
    t = text or ""
    words = LEXICON.get(label, [])
    for w in words:
        idx = t.find(w)
        if idx != -1:
            return t[max(0, idx-20):min(len(t), idx+len(w)+20)].strip()
    return t[:max_len].strip()

### 6. Prompt with few-shot examples

In [ ]:
CONTEXT_KNOWLEDGE = """以下是相关景点的背景知识，请在标注时参考：
1. 颐和园：中国清朝时期皇家园林，前身为清漪园，位于北京市西郊海淀区新建宫门路19号。其利用昆明湖、万寿山为基址，以杭州西湖风景为蓝本，汲取江南园林的某些设计手法和意境而建成的一座大型天然山水园，也是保存得最完整的一座皇家行宫御苑，被誉为皇家园林博物馆。全园占地3.009平方千米，水面约占四分之三，由万寿山、昆明湖组成。园中分布着点景建筑物百余座、大小院落二十余处，古建筑三千余间，面积近7万平方米。古树名木一千六百余株。
颐和园始建于清乾隆十五年（1750年），咸丰十年（1860年）被英法联军烧毁。光绪十二年（1886年），清廷挪用海军经费等款项开始重建，并于两年后取用今名，作为慈禧太后的颐养之所。光绪二十六年（1900年）又遭八国联军破坏，光绪二十八年（1902年）修复。
颐和园最突出的特色在于山环水抱的自然景观和皇家气派的宫苑建筑的完美结合，在真山、真水中进行景点设计，既突出皇家的气派，又体现了园林与山、水的和谐统一；既师法自然又高于自然，将自然美与人工雕琢巧妙地融于一体。颐和园的园林造景，融合了中国绘画、诗歌和文学的意境，造就出中国皇家园林宏大的气势、辉煌的色彩及其与山水绿化环境的充分协调关系。颐和园藏文物有四万余件，品类几乎涵盖了中国传世文物的所有门类，国家级文物有二万余件。
1961年3月4日，颐和园被中华人民共和国国务院公布为第一批全国重点文物保护单位。1998年12月，颐和园被联合国教科文组织列入《世界遗产名录》。
2. 圆明园：中国清代大型皇家园林，国家AAAAA级旅游景区，位于北京市海淀区清华西路28号，占地350多公顷，其中水面面积约140公顷，由圆明园、绮春园、长春园组成，而以圆明园最大，故统称圆明园（亦称圆明三园）。圆明园不仅汇集了江南若干名园胜景，还移植了西方园林建筑，集当时古今中外造园艺术之大成。堪称人类文化的宝库之一，是当时世界上最大的一座博物馆。
康熙末年和雍正年间，圆明园开始兴建。清雍正二年（1724年），圆明园的扩建工程正式开始。乾隆帝继位后，在圆明园内调整了园林的景观，增添了建筑组群。1860年，英法侵略者纵火焚烧圆明园，圆明园及附近的清漪园、静明园、静宜园、畅春园及海淀镇均被烧。清光绪二十六年（1900年），八国联军侵占北京，圆明园的建筑和古树名木遭到彻底毁灭。清朝灭亡后，圆明园的遗物又长期遭到官僚、军阀、奸商巧取豪夺。建国后，中国政府对圆明园开始了保护整修工作。
圆明园大量仿建了中国各地特别是江南的许多名园胜景，其主要建筑类型包括殿、堂、亭、台、楼、阁、榭、廊、轩、斋、房、舫、馆、厅、桥、闸、墙、塔，以及寺庙、道观、村居、街市等；建筑平面布局共有38种。圆明园的园林造景多以水为主题，部分建筑包含大量宗教元素。
3. 香山公园：香山公园（Fragrant Hills Park），位于北京市海淀区买卖街40号，北京市区西北郊，占地188公顷，是一座具有山林特色的皇家园林。早在元、明、清时，皇家就在香山营建离宫别院，每逢夏秋时节皇帝都要到此狩猎纳凉。咸丰十年（1860年）和光绪二十六年（1900年）先后两次被英法联军、八国联军焚毁，1956年开辟为人民公园。
香山公园地处西山东坡之腹心，也是西山山梁东端的枢纽，峰峦层叠、涧壑交错。有香山寺、洪光寺、双清别墅等著名旅游景点。景区内主峰香炉峰俗称“鬼见愁”，海拔575米。公园树木繁多，森林覆盖率高达96%，具有独特的“山川、名泉、古树、红叶”丰富的园林内涵。

以下是各个 CES 类别的详细定义，请在标注时参考：
*   **Spiritual_and_religious_value**：许多宗教赋予生态系统或其组成部分精神和宗教价值。
*   **Knowledge_and_education**：生态系统及其组成部分和过程为许多社会的正规和非正规教育提供了基础。同时，生态系统也影响了不同文化所发展的知识体系类型。
*   **Inspiration**：生态系统为艺术、民间传说、民族象征、建筑和广告提供了丰富的灵感来源。
*   **Aesthetic_values_nature**：许多人发现生态系统各个方面的美感或审美价值，这体现在对公园、风景优美路线的支持以及住宅选址上。
*   **Aesthetic_values_architecture**：许多人发现生态系统各个方面的美感或审美价值，这体现在对公园、风景优美路线的支持以及住宅选址上。（注意：此定义与 'Aesthetic_values_nature' 共享，实际分类时侧重建筑美）
*   **Aesthetic_object_landscape**：许多人发现生态系统各个方面的美感或审美价值，这体现在对公园、风景优美路线的支持以及住宅选址上。（注意：此定义与 'Aesthetic_values_nature' 共享，实际分类时侧重整体景观美）
*   **Social_relations**：生态系统影响特定文化中建立的社会关系类型。例如，渔业社会的社会关系在许多方面不同于游牧畜牧或农业社会。
*   **Sense_of_place**：许多人珍视与环境公认特征（包括生态系统方面）相关的“地方感”。
*   **Cultural_heritage_values**：许多社会高度重视维护具有历史意义的景观（“文化景观”）或具有文化意义的物种。
*   **Recreation_and_visitation**：人们通常根据特定区域的自然或人造景观特征来选择休闲地点。"""

FEW_SHOT_SECTION = """以下是标注示例：

示例 1
评论：
（桂月飘香）2009年，北京旅游，回忆篇。文昌阁，是一座像城楼一样的建筑，是颐和园内六座城关建筑中最大的一座，始建于公元1750年，1860年被英法联军入侵时候烧毁。光绪年间，文昌阁又进行了重建，主阁两层，内供铜铸文昌帝君和仙童。文昌阁与万寿山西贡武圣的宿云檐象征“文物辅弼”。可以上去参观，从楼上远眺，颐和园的精致美景尽收眼底，只是游客太多，拍出的风景照片都很无奈啊。
输出：
Inspiration\t0\t拍出的风景照片都很无奈
Aesthetic_values_architecture\t1\t像城楼一样的建筑
Aesthetic_object_landscape\t1\t颐和园的精致美景
Cultural_heritage_values\t1\t始建于公元1750年
Recreation_and_visitation\t1\t可以上去参观
PRIMARY\tAesthetic_object_landscape
CONF\t1.0

示例 2
评论：
可以准备一天时间去慢慢逛，里面厕所很多。面积很大。
输出：
Recreation_and_visitation\t1\t慢慢逛
PRIMARY\tRecreation_and_visitation
CONF\t1.0

示例 3
评论：
原本是个市民休闲放风筝的理想集结地，如今为了所谓的面子工程，某些人的官威，成了一个“保安比花还多”的公园。
输出：
Recreation_and_visitation\t0\t市民休闲放风筝的理想集结地
PRIMARY\tRecreation_and_visitation
CONF\t0.8

示例 4
评论：
德和园建于清光绪年间，被誉为天下第一大戏楼，是专供慈禧太后看戏的场所。德和园见证了晚清皇家的宫廷奢华生活，也展现了古代工匠的高超技艺。今天可真真切切地感受到浓郁的历史文化氛围，耳闻不如目睹，欢迎大家来打卡。
输出：
Aesthetic_values_architecture\t1\t天下第一大戏楼
Cultural_heritage_values\t1\t清光绪年间
Recreation_and_visitation\t1\t欢迎大家来打卡
PRIMARY\tCultural_heritage_values
CONF\t1.0

示例 5
评论：
2024/1/10（周三）中午走到这里的。在这里，需要看简介也需要有丰富的想象力。水面结冰，码头附近有几只绿头鸭跟黑天鹅。在这里很舒服，看着远远的小桥，还有两堆石块堆。冰上放着“请勿踏冰"的牌子。方壶胜境建自乾隆三年前后，取材于道家的“仙山琼阁”意境。而我只有满眼的白色冰雪跟想象力还有冷。
输出：
Spiritual_and_religious_value\t1\t很舒服
Inspiration\t1\t需要有丰富的想象力
Aesthetic_values_architecture\t1\t小桥
Aesthetic_object_landscape\t1\t满眼的白色冰雪
Cultural_heritage_values\t1\t仙山琼阁
PRIMARY\tAesthetic_values_architecture
CONF\t0.9

示例 6
评论：
风景非常差 普的不能再普。没必要再去第二次，绝对不会再去第二次。古迹也都是fake的。北边太陡了，山顶也没风景。后来六点从上往下下，真怕天黑了，吓死。中路走到一楼的时候发现有很多大松柏还不错。
输出：
Spiritual_and_religious_value\t0\t吓死
Aesthetic_values_nature\t1\t大松柏还不错
Aesthetic_object_landscape\t0\t风景非常差
Cultural_heritage_values\t0\t古迹都是fake的
PRIMARY\tAesthetic_object_landscape
CONF\t0.8

示例 7
评论：
可以带孩子一起去爬山的公园。香山公园山头不算太高，但是空气清新可以让人充分体验到身心放松的松弛感。而且带孩子一起去可以一边观景，一边了解属于这个公园的历史，既可以放松还可以学知识。
输出：
Spiritual_and_religious_value\t1\t身心放松
Knowledge_and_education\t1\t了解历史
Aesthetic_object_landscape\t1\t观景
Social_relations\t1\t带孩子一起
Recreation_and_visitation\t1\t爬山
PRIMARY\tRecreation_and_visitation
CONF\t1.0

示例 8
评论：
广润灵雨祠俗称龙王庙，是颐和园南湖岛上的重要建筑。始建于明代，乾隆年间重修。乾隆、嘉庆、道光、咸丰诸帝都曾来此上香祈雨。
输出：
Spiritual_and_religious_value\t1\t上香祈雨
Aesthetic_values_architecture\t1\t重要建筑
Cultural_heritage_values\t1\t乾隆年间
PRIMARY\tCultural_heritage_values
CONF\t1.0

示例 9
评论：
雪中的乐寿堂更有一番风景。慈禧有几张在颐和园冬天拍摄的照片，现在配上一点小雪，更加有代入感。
输出：
Aesthetic_values_nature\t1\t小雪
Aesthetic_values_architecture\t1\t乐寿堂
Sense_of_place\t1\t代入感
Cultural_heritage_values\t1\t慈禧
PRIMARY\tSense_of_place
CONF\t0.9

示例 10
评论：
非常适合带孩子来，夏天沿着小溪的步行道非常凉爽，还可以带水枪玩水。适合给孩子做自然科普，孩子玩的很开心。爬山也是个好选择。
输出：
Spiritual_and_religious_value\t1\t玩的很开心
Knowledge_and_education\t1\t自然科普
Aesthetic_values_nature\t1\t小溪
Social_relations\t1\t带孩子
Recreation_and_visitation\t1\t爬山
PRIMARY\tSocial_relations
CONF\t1.0

示例 11
评论：
暑假带着孩子家人去了一次颐和园，十七孔桥壮观雄伟。站在桥上看风景拍照非常不错，非常值得一来。以后有机会还会和家人朋友再去逛逛。
输出：
Inspiration\t1\t看风景拍照
Aesthetic_values_architecture\t1\t十七孔桥壮观雄伟
Aesthetic_object_landscape\t1\t看风景
Social_relations\t1\t家人朋友
Recreation_and_visitation\t1\t逛逛
PRIMARY\tRecreation_and_visitation
CONF\t1.0
"""

SYSTEM = (
    "你是中文游客评论的 CES 标注员。\n"
    f"{CONTEXT_KNOWLEDGE}\n"
    "请注意：在对'Spiritual_and_religious_value'类别进行标注时，要根据语境判断，避免将'老佛爷'、'佛香阁'等作为特有名词出现的词语误判为宗教或精神价值。\n"
    "只输出纯文本，不要 JSON，不要解释。\n"
    "输出格式（每行一条）：\n"
    "label\\tsentiment\\tevidence\n"
    "sentiment: 1 为正向，0 为负向/中性。\n"
    "最后两行必须是：\n"
    "PRIMARY\\t<一个label或空>\n"
    "CONF\\t<0到1的小数>\n"
    "label 只能从以下列表选：\n"
    f"{CES_CLASSES}\n"
    "evidence 不超过80字，必须来自评论，且不要包含制表符和换行。\n"
    "如果没有任何 CES：不要输出任何 label 行，PRIMARY 为空，CONF=0。\n"
    f"{FEW_SHOT_SECTION}"
)

def build_user_prompt(text: str, hints: list):
    return (
        "请根据背景知识并模仿上述示例，对以下评论进行标注：\n"
        f"评论：{text}\n\n"
        f"词库提示标签（优先考虑）：{hints}"
    )

### 7. Utility functions

In [ ]:
# =========================
# 7. Utility functions
# =========================
def _throttle():
    global _last_call
    now = time.time()
    wait = MIN_INTERVAL_SEC - (now - _last_call)
    if wait > 0:
        time.sleep(wait)
    _last_call = time.time()

def make_comment_id(text: str):
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def guess_text_col(df: pd.DataFrame):
    # Prioritize columns most likely to contain actual review text
    candidates = ["评论内容", "comment_detail", "content", "评论", "comment", "review", "text", "内容", "评价"]
    for c in candidates:
        if c in df.columns:
            # Ensure the column actually contains string-like data (not just empty or NaNs)
            if df[c].astype(str).str.strip().str.len().sum() > 0:
                return c

    # Fallback to general object columns, but exclude image-related ones
    obj_cols = [c for c in df.columns if df[c].dtype == "object" and "图片" not in c and "image" not in c.lower()]
    if not obj_cols:
        return None
    lens = {c: df[c].astype(str).map(len).mean() for c in obj_cols if df[c].astype(str).str.strip().str.len().sum() > 0}
    if lens:
        return max(lens, key=lens.get)
    return None

def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="gbk")

def extract_raw_from_response(data: dict) -> str:
    'Concatenate all text from candidates -> parts.'
    try:
        c0 = (data.get("candidates") or [])[0]
        content = c0.get("content") or {}
        parts = content.get("parts") or []
    except Exception:
        return ""
    texts = []
    for p in parts:
        if isinstance(p, dict) and isinstance(p.get("text"), str) and p["text"].strip():
            texts.append(p["text"].strip())
    return "\n".join(texts).strip()

def parse_model_output_to_obj(raw: str) -> dict:
    '\n    Convert the model output to a dict where possible:\n    - Find the {...} block.\n    - Try json.loads(strict=False).\n    - If parsing fails, apply minimal cleanup.\n    - As a last resort, extract label/evidence/primary/conf with regular expressions.\n    '
    s = (raw or "").strip()
    s = s.replace("```json", "").replace("```", "").strip()

    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j > i:
        s = s[i:j+1]

    # Normalise Chinese quotation marks.
    s = s.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

    # Try parsing as JSON.
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return json.loads(s, strict=False)
    except Exception:
        pass

    # Remove trailing commas.
    s2 = re.sub(r",\s*([}{])", r"\1", s)
    try:
        return json.loads(s2, strict=False)
    except Exception:
        pass

    # Fall back to regular expressions.
    labels = re.findall(r'"label"\s*:\s*"([^"]+)"', s2)
    evidences = re.findall(r'"evidence"\s*:\s*"([^"]*)"', s2)
    le = []
    for lab, ev in zip(labels, evidences):
        le.append({"label": lab.strip(), "evidence": (ev or "").strip()})

    m = re.search(r'"primary_label"\s*:\s*"([^"]*)"', s2)
    primary = (m.group(1).strip() if m else "")

    m2 = re.search(r'"confidence"\s*:\s*([0-9]*\.?[0-9]+)', s2)
    conf = float(m2.group(1)) if m2 else 0.0

    return {"label_evidence": le, "primary_label": primary, "confidence": conf}

### 8. SQLite cache

In [ ]:
# =========================
# 8. SQLite cache
# =========================
def init_db(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA busy_timeout=8000;")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS responses (
      comment_id TEXT PRIMARY KEY,
      status TEXT,
      response_json TEXT,
      updated_at TEXT
    )
    """)
    conn.commit()
    return conn

def cache_has_ok(cur, cid: str):
    cur.execute("SELECT status FROM responses WHERE comment_id=?", (cid,))
    row = cur.fetchone()
    return bool(row) and row[0] == "ok"

def cache_set(cur, cid: str, status: str, obj):
    cur.execute("""
      INSERT INTO responses(comment_id,status,response_json,updated_at)
      VALUES(?,?,?,?)
      ON CONFLICT(comment_id) DO UPDATE SET
        status=excluded.status,
        response_json=excluded.response_json,
        updated_at=excluded.updated_at
    """, (cid, status, json.dumps(obj, ensure_ascii=False), datetime.now().isoformat(timespec="seconds")))

### 9. Core API call with stability safeguards

In [ ]:
# =========================
# 9. Core API call with enhanced parsing and lexicon fallback
# =========================
def parse_tsv_output(raw: str):
    raw = (raw or "").strip()
    raw = raw.replace('```tsv', '').replace('```', '').strip()
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    label_evidence = []
    primary, conf = "", 0.0
    for ln in lines:
        parts = re.split(r'\t+', ln)
        if len(parts) < 2:
            parts = re.split(r'\s{2,}', ln)
            if len(parts) < 2: continue
        k = parts[0].strip().upper()
        if k == "PRIMARY": primary = parts[1].strip()
        elif k == "CONF":
            try:
                match = re.search(r"[0-9.]+", parts[1])
                conf = float(match.group()) if match else 0.0
            except: conf = 0.0
        elif parts[0].strip() in CES_CLASSES:
            label = parts[0].strip()
            sentiment = 1
            evidence = ""
            if len(parts) >= 3:
                try: sentiment = int(re.search(r"\d", parts[1]).group())
                except: sentiment = 1
                evidence = parts[2].strip()
            elif len(parts) == 2:
                evidence = parts[1].strip()
            label_evidence.append({"label": label, "sentiment": sentiment, "evidence": evidence[:80]})
    seen = set()
    cleaned = [item for item in label_evidence if not (item["label"] in seen or seen.add(item["label"])) ]
    return {"label_evidence": cleaned, "primary_label": primary, "confidence": conf}

def call_api(text: str, hints: list) -> dict:
    payload = {
        "contents": [{"parts": [{"text": SYSTEM}, {"text": build_user_prompt(text, hints)}]}],
        "generationConfig": {"temperature": 0, "maxOutputTokens": 800}
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            _throttle()
            r = requests.post(GEMINI_URL, headers=HEADERS, params={"key": API_KEY}, data=json.dumps(payload, ensure_ascii=False), timeout=TIMEOUT)
            if r.status_code != 200: raise RuntimeError(f"HTTP {r.status_code}")
            data = r.json()
            raw = extract_raw_from_response(data)
            if not raw: raise RuntimeError("Empty Response")

            obj = parse_tsv_output(raw)

            # --- Lexicon fallback logic ---
            # Use the lexicon hints when available if the model returns no result or low confidence.
            if hints and (not obj["label_evidence"] or obj["confidence"] < MIN_CONFIDENCE_FOR_FALLBACK):
                fallback_items = []
                for h in hints:
                    if h in CES_CLASSES:
                        fallback_items.append({"label": h, "sentiment": 1, "evidence": evidence_from_text(h, text)})
                if fallback_items:
                    obj["label_evidence"] = fallback_items
                    obj["confidence"] = max(obj["confidence"], 0.35) # Assign a fixed confidence to fallback results.
            return obj
        except Exception as e:
            print(f"Attempt {attempt} failed: {e}")
            time.sleep(2)
    return {"label_evidence": [], "primary_label": "", "confidence": 0.0, "_error": "ALL_ATTEMPTS_FAILED"}
print("call_api function loaded.") # Added for debugging

### 10. Main workflow

In [ ]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime

INPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/源数据-重分类/公园")
OUTPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/评论-分析结果/公园")

# Helper to read CSV robustly (copied from main processing cell)
def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="gbk")

# Helper to extract timestamp from filename for sorting (copied from main processing cell)
def get_timestamp_from_filename(filepath: Path) -> datetime:
    match = re.search(r'-(\d{8}_\d{4})\.csv$', filepath.name)
    if match:
        return datetime.strptime(match.group(1), '%Y%m%d_%H%M')
    return datetime.min # Return smallest possible datetime if no timestamp found

def make_comment_id(text: str): # Moved from new_cell_14 to be available for validation
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def guess_text_col(df: pd.DataFrame):
    candidates = ["content", "评论", "comment", "review", "text", "内容", "评价", "点评内容", "comment_detail"]
    for c in candidates:
        if c in df.columns:
            return c
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        return None
    lens = {c: df[c].astype(str).map(len).mean() for c in obj_cols}
    return max(lens, key=lens.get)


# Dictionary to store input file base name -> full path -> set of expected comment_ids
input_files_expected_cids = {}
for file_path in INPUT_BASE_DIR.glob('**/*.csv'):
    if file_path.parent == INPUT_BASE_DIR:
        base_name = file_path.stem
    else:
        base_name = file_path.parent.name

    try:
        df_input = read_csv_robust(file_path)
        text_col = guess_text_col(df_input)
        if text_col is None:
            print(f"警告: 输入文件 {file_path.name} 未找到合适的文本列，跳过此文件进行完整性检查。")
            continue

        expected_cids = set()
        for text_content in df_input[text_col].dropna().astype(str):
            if text_content.strip():
                expected_cids.add(make_comment_id(text_content.strip()))

        input_files_expected_cids[base_name] = {"path": file_path, "expected_cids": expected_cids}

    except Exception as e:
        print(f"警告: 无法读取输入文件 {file_path.name} ({e})，将跳过此文件进行完整性检查。")


# Dictionary to store output file base name -> latest output path and its timestamp and actual cids
output_latest_files = {}
for file_path in OUTPUT_BASE_DIR.glob('*.csv'):
    match = re.search(r"^(.*?)-\d{8}_\d{4}\.csv$", file_path.name)
    base_name = match.group(1) if match else file_path.stem

    current_timestamp = get_timestamp_from_filename(file_path)

    if base_name not in output_latest_files or current_timestamp > output_latest_files[base_name]["timestamp"]:
        output_latest_files[base_name] = {"path": file_path, "timestamp": current_timestamp, "actual_cids": set()} # Initialize actual_cids

# Now, read the comment_ids for the latest output files
for base_name, info in output_latest_files.items():
    try:
        df_output = read_csv_robust(info["path"])
        if "comment_id" in df_output.columns:
            output_latest_files[base_name]["actual_cids"] = set(df_output["comment_id"].dropna().astype(str).tolist())
        else:
            print(f"警告: 输出文件 {info['path'].name} 中未找到 'comment_id' 列，无法进行精确完整性检查。")
            output_latest_files[base_name]["actual_cids"] = set() # Treat as empty if comment_id not found

    except Exception as e:
        print(f"警告: 无法读取输出文件 {info['path'].name} ({e})，该文件将标记为未完全处理。")
        output_latest_files[base_name]["actual_cids"] = set() # If read fails, assume no CIDs


missing_output_files = []
incomplete_processing_files = []
processed_complete_files = []

for input_base_name, input_data in input_files_expected_cids.items():
    expected_cids = input_data["expected_cids"]

    if input_base_name not in output_latest_files:
        if expected_cids: # Only report missing if there are actual comments to process
            missing_output_files.append(input_base_name)
        else: # If input has no valid comments, consider it complete if no output needed
            processed_complete_files.append(input_base_name + " (无有效评论)")

    else:
        output_data = output_latest_files[input_base_name]
        actual_cids = output_data["actual_cids"]

        # Check if all expected comment_ids are present in the actual output comment_ids
        if not expected_cids.issubset(actual_cids): # If actual_cids doesn't contain all expected_cids
            missing_count = len(expected_cids - actual_cids)
            incomplete_processing_files.append(
                f"{input_base_name} (缺失 {missing_count} 条评论ID；输入预期总数: {len(expected_cids)}, 输出实际总数: {len(actual_cids)} - {output_data['path'].name})")
        else:
            processed_complete_files.append(input_base_name)

if not missing_output_files and not incomplete_processing_files:
    print("✅ 所有输入文件都在输出文件夹中找到了对应的结果文件，且评论ID匹配。")
    print(f"共计 {len(processed_complete_files)} 个文件已完全处理。")
else:
    if missing_output_files:
        print("❌ 以下输入文件尚未在输出文件夹中找到对应的结果文件：")
        for filename in sorted(missing_output_files):
            print(f"- {filename}.csv")
    if incomplete_processing_files:
        print("\n⚠️ 以下文件可能未完全处理（输出文件评论ID不完整或输出文件读取失败）：")
        for filename_info in sorted(incomplete_processing_files):
            print(f"- {filename_info}")
    print("\n请重新运行主流程单元格 (cell_id: new_cell_20) 来处理这些文件。")

In [ ]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime
import hashlib
import time
import random
import sqlite3
import json
import requests

from google import genai
from google.genai import types

# =========================
# 2. API configuration for Vertex AI using the latest google-genai version
# =========================
# Enter your project ID here.
PROJECT_ID = "api-try-495100"
LOCATION = "us-central1"

# Initialise the client with the current official 2026 approach.
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

# =========================
# 3. Performance and reliability settings
# =========================
MIN_INTERVAL_SEC = 3.0 # Increased to reduce base API call frequency
_last_call = 0.0

WRITE_EVERY = 30              # Number of records processed between CSV saves.
TEXT_MAX_CHARS = 900          # Truncate long reviews to improve stability and limit latency and cost.
MAX_RETRIES = 5               # Maximum retries per record.
MAX_TOTAL_SEC_PER_ITEM = 90   # Maximum total processing time per record.
MIN_CONFIDENCE_FOR_FALLBACK = 0.6 # Consider a lexicon fallback if the model returns confidence below this value.

# =========================
# 4. CES categories and output columns for evidence and sentiment
# =========================
CES_CLASSES = [
    "Spiritual_and_religious_value",
    "Knowledge_and_education",
    "Inspiration",
    "Aesthetic_values_nature",
    "Aesthetic_values_architecture",
    "Aesthetic_values_landscape",
    "Social_relations",
    "Sense_of_place",
    "Cultural_heritage_values",
    "Recreation_and_visitation"
]

CES_DATA_COLS = []
for c in CES_CLASSES:
    CES_DATA_COLS.append(f"EV_{c}")
    CES_DATA_COLS.append(f"SNT_{c}")

# =========================
# 5. Lexicon
# =========================
LEXICON = {
    "Sense_of_place": ["氛围","感觉","意境","代入感","沉浸","身临其境","古韵","古风","古意","历史氛围","文化氛围","园林气息","皇家气息","古典气息","气息","地方感","归属感","熟悉感","亲切感","老北京","北京味道","京味","京味文化"],
    "Spiritual_and_religious_value": ["祈福","祈愿","祈祷","许愿","虔诚","上香","祠","寺庙","庙宇","佛","菩萨","道观","神圣","信仰","宗教","香火","拜佛","烧香","供奉","求签","祭祀","祭祠","宁静","安静","清幽","幽静","静谧","安逸","放松","治愈","惬意","舒适","悠闲","松弛"],
    "Knowledge_and_education": ["知识","科普","讲解","讲解员","解说","了解","学习","介绍","说明","展览","展厅","展馆","博物馆","历史介绍","文化讲解","教育","孩子","学生","研学","课堂","长见识"],
    "Inspiration": ["灵感","启发","激发","创作","思考","感悟","感想","触动","诗意","写生","画画","摄影创作"],
    "Aesthetic_values_nature": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","景色","风景","美景","景致","风光","景观","园林美","画面","视觉","色彩","光影","倒影","构图","层次","景观设计","拍照","摄影","出片","拍照好看","适合拍照","打卡拍照","夜景","灯光","日落","日出","夕阳","晨光","树","树木","树林","森林","古树","绿树","绿荫","花","花海","花园","花坛","荷花","樱花","梅花","草","草地","草坪","湖","湖水","湖面","池塘","水面","溪流","荷塘","水景"],
    "Aesthetic_values_architecture": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","建筑","亭","台","楼","阁","殿","宫","院","廊","长廊","回廊","桥","石桥","拱桥","小桥","门","宫门","山门","牌楼","牌坊","碑","石碑","碑刻","寺","庙","观","祠","古建","古建筑"],
    "Aesthetic_values_landscape": ["美丽","漂亮","好看","很美","太美","绝美","惊艳","震撼","壮观","赏心悦目","园林","公园","景区","景点","整体景观","园子","园子里"],
    "Social_relations": ["朋友","家人","亲子","情侣","一起","陪伴","同行","合影","拍合照","约会","聚会","团建","带父母","带孩子"],
    "Cultural_heritage_values": ["历史","古迹","文物","文化","传统","遗产","皇家","宫廷","园林文化","历史感","年代感","历史厚重","文化底蕴","文化气息","文化价值","保护","修复","修缮","保护文物","保护遗址","传承","历史价值","亭","台","楼","阁","殿","宫","院","园","廊","长廊","回廊","桥","门","牌楼","牌坊","碑","石碑","碑刻","题字","匾额","寺","庙","观","祠","遗址","遗迹","城墙","古建","古建筑"],
    "Recreation_and_visitation": ["游玩","逛","逛园","逛逛","游览","参观","看看","散步","溜达","遛弯","打卡","打卡点","网红打卡","旅游","旅行","景点","游客","很多人来","推荐","值得去","值得一看","排队","门票","慢慢逛","随便逛"]
}

EXCLUDE_FROM_SPIRITUAL = ["老佛爷", "佛香阁"]

def lexicon_hint_labels(text: str):
    t = text or ""
    hits = set()
    for lab, words in LEXICON.items():
        if lab == "Spiritual_and_religious_value":
            if any(ex_word in t for ex_word in EXCLUDE_FROM_SPIRITUAL):
                continue
        if any(w in t for w in words):
            hits.add(lab)
    return [c for c in CES_CLASSES if c in hits]

def evidence_from_text(label: str, text: str, max_len: int = 80) -> str:
    t = text or ""
    words = LEXICON.get(label, [])
    for w in words:
        idx = t.find(w)
        if idx != -1:
            return t[max(0, idx-20):min(len(t), idx+len(w)+20)].strip()
    return t[:max_len].strip()

# =========================
# ⑥ Prompt
# =========================
CONTEXT_KNOWLEDGE = """以下是相关景点的背景知识，请在标注时参考：
1. 颐和园：中国清朝时期皇家园林，前身为清漪园，位于北京市西郊海淀区新建宫门路19号。其利用昆明湖、万寿山为基址，以杭州西湖风景为蓝本，汲取江南园林的某些设计手法和意境而建成的一座大型天然山水园，也是保存得最完整的一座皇家行宫御苑，被誉为皇家园林博物馆。全园占地3.009平方千米，水面约占四分之三，由万寿山、昆明湖组成。园中分布着点景建筑物百余座、大小院落二十余处，古建筑三千余间，面积近7万平方米。古树名木一千六百余株。
颐和园始建于清乾隆十五年（1750年），咸丰十年（1860年）被英法联军烧毁。光绪十二年（1886年），清廷挪用海军经费等款项开始重建，并于两年后取用今名，作为慈禧太后的颐养之所。光绪二十六年（1900年）又遭八国联军破坏，光绪二十八年（1902年）修复。
颐和园最突出的特色在于山环水抱的自然景观和皇家气派的宫苑建筑的完美结合，在真山、真水中进行景点设计，既突出皇家的气派，又体现了园林与山、水的和谐统一；既师法自然又高于自然，将自然美与人工雕琢巧妙地融于一体。颐和园的园林造景，融合了中国绘画、诗歌和文学的意境，造就出中国皇家园林宏大的气势、辉煌的色彩及其与山水绿化环境的充分协调关系。颐和园藏文物有四万余件，品类几乎涵盖了中国传世文物的所有门类，国家级文物有二万余件。
1961年3月4日，颐和园被中华人民共和国国务院公布为第一批全国重点文物保护单位。1998年12月，颐和园被联合国教科文组织列入《世界遗产名录》。
2. 圆明园：中国清代大型皇家园林，国家AAAAA级旅游景区，位于北京市海淀区清华西路28号，占地350多公顷，其中水面面积约140公顷，由圆明园、绮春园、长春园组成，而以圆明园最大，故统称圆明园（亦称圆明三园）。圆明园不仅汇集了江南若干名园胜景，还移植了西方园林建筑，集当时古今中外造园艺术之大成。堪称人类文化的宝库之一，是当时世界上最大的一座博物馆。
康熙末年和雍正年间，圆明园开始兴建。清雍正二年（1724年），圆明园的扩建工程正式开始。乾隆帝继位后，在圆明园内调整了园林的景观，增添了建筑组群。1860年，英法侵略者纵火焚烧圆明园，圆明园及附近的清漪园、静明园、静宜园、畅春园及海淀镇均被烧。清光绪二十六年（1900年），八国联军侵占北京，圆明园的建筑和古树名木遭到彻底毁灭。清朝灭亡后，圆明园的遗物又长期遭到官僚、军阀、奸商巧取豪夺。建国后，中国政府对圆明园开始了保护整修工作。
圆明园大量仿建了中国各地特别是江南的许多名园胜景，其主要建筑类型包括殿、堂、亭、台、楼、阁、榭、廊、轩、斋、房、舫、馆、厅、桥、闸、墙、塔，以及寺庙、道观、村居、街市等；建筑平面布局共有38种。圆明园的园林造景多以水为主题，部分建筑包含大量宗教元素。
3. 香山公园：香山公园（Fragrant Hills Park），位于北京市海淀区买卖街40号，北京市区西北郊，占地188公顷，是一座具有山林特色的皇家园林。早在元、明、清时，皇家在香山营建离宫别院，每逢夏秋时节皇帝都要到此狩猎纳凉。咸丰十年（1860年）和光绪二十六年（1900年）先后两次被英法联军、八国联军焚毁，1956年开辟为人民公园。
香山公园地处西山东坡之腹心，也是西山山梁东端的枢纽，峰峦层叠、涧壑交错。有香山寺、洪光寺、双清别墅等著名旅游景点。景区内主峰香炉峰俗称“鬼见愁”，海拔575米。公园树木繁多，森林覆盖率高达96%，具有独特的“山川、名泉、古树、红叶”丰富的园林内涵。"""

FEW_SHOT_SECTION = """以下是标注示例：

示例 1
评论：
（桂月飘香）2009年，北京旅游，回忆篇。文昌阁，是一座像城楼一样的建筑，是颐和园内六座城关建筑中最大的一座，始建于公元1750年，1860年被英法联军入侵时候烧毁。光绪年间，文昌阁又进行了重建，主阁两层，内供铜铸文昌帝君和仙童。文昌阁与万寿山西贡武圣的宿云檐象征“文物辅弼”。可以上去参观，从楼上远眺，颐和园的精致美景尽收眼底，只是游客太多，拍出的风景照片都很无奈啊。
输出：
Inspiration	0	拍出的风景照片都很无奈
Aesthetic_values_architecture	1	像城楼一样的建筑
Aesthetic_object_landscape	1	颐和园的精致美景
Cultural_heritage_values	1	始建于公元1750年
Recreation_and_visitation	1	可以上去参观
PRIMARY	Aesthetic_object_landscape
CONF	1.0

示例 2
评论：
可以准备一天时间去慢慢逛，里面厕所很多。面积很大。
输出：
Recreation_and_visitation	1	慢慢逛
PRIMARY	Recreation_and_visitation
CONF	1.0

示例 3
评论：
原本是个市民休闲放风筝的理想集结地，如今为了所谓的面子工程，某些人的官威，成了一个“保安比花还多”的公园。
输出：
Recreation_and_visitation	0	市民休闲放风筝的理想集结地
PRIMARY	Recreation_and_visitation
CONF	0.8

示例 4
评论：
德和园建于清光绪年间，被誉为天下第一大戏楼，是专供慈禧太后看戏的场所。德和园见证了晚清皇家的宫廷奢华生活，也展现了古代工匠的高超技艺。今天可真真切切地感受到浓郁的历史文化氛围，耳闻不如目睹，欢迎大家来打卡。
输出：
Aesthetic_values_architecture	1	天下第一大戏楼
Cultural_heritage_values	1	清光绪年间
Recreation_and_visitation	1	欢迎大家来打卡
PRIMARY	Cultural_heritage_values
CONF	1.0

示例 5
评论：
2024/1/10（周三）中午走到这里的。在这里，需要看简介也需要有丰富的想象力。水面结冰，码头附近有几只绿头鸭跟黑天鹅。在这里很舒服，看着远远的小桥，还有两堆石块堆。冰上放着“请勿踏冰"的牌子。方壶胜境建自乾隆三年前后，取材于道家的“仙山琼阁”意境。而我只有满眼的白色冰雪跟想象力还有冷。
输出：
Spiritual_and_religious_value	1	很舒服
Inspiration	1	需要有丰富的想象力
Aesthetic_values_architecture	1	小桥
Aesthetic_object_landscape	1	满眼的白色冰雪
Cultural_heritage_values	1	仙山琼阁
PRIMARY	Aesthetic_values_architecture
CONF	0.9

示例 6
评论：
风景非常差 普的不能再普。没必要再去第二次，绝对不会再去第二次。古迹也都是fake的。北边太陡了，山顶也没风景。后来六点从上往下下，真怕天黑了，吓死。中路走到一楼的时候发现有很多大松柏还不错。
输出：
Spiritual_and_religious_value	0	吓死
Aesthetic_values_nature	1	大松柏还不错
Aesthetic_object_landscape	0	风景非常差
Cultural_heritage_values	0	古迹都是fake的
PRIMARY	Aesthetic_object_landscape
CONF	0.8

示例 7
评论：
可以带孩子一起去爬山的公园。香山公园山头不算太高，但是空气清新可以让人充分体验到身心放松的松弛感。而且带孩子一起去可以一边观景，一边了解属于这个公园的历史，既可以放松还可以学知识。
输出：
Spiritual_and_religious_value	1	身心放松
Knowledge_and_education	1	了解历史
Aesthetic_object_landscape	1	观景
Social_relations	1	带孩子一起
Recreation_and_visitation	1	爬山
PRIMARY	Recreation_and_visitation
CONF	1.0

示例 8
评论：
广润灵雨祠俗称龙王庙，是颐和园南湖岛上的重要建筑。始建于明代，乾隆年间重修。乾隆、嘉庆、道光、咸丰诸帝都曾来此上香祈雨。
输出：
Spiritual_and_religious_value	1	上香祈雨
Aesthetic_values_architecture	1	重要建筑
Cultural_heritage_values	1	乾隆年间
PRIMARY	Cultural_heritage_values
CONF	1.0

示例 9
评论：
雪中的乐寿堂更有一番风景。慈禧有几张在颐和园冬天拍摄的照片，现在配上一点小雪，更加有代入感。
输出：
Aesthetic_values_nature	1	小雪
Aesthetic_values_architecture	1	乐寿堂
Sense_of_place	1	代入感
Cultural_heritage_values	1	慈禧
PRIMARY	Sense_of_place
CONF	0.9

示例 10
评论：
非常适合带孩子来，夏天沿着小溪的步行道非常凉爽，还可以带水枪玩水。适合给孩子做自然科普，孩子玩的很开心。爬山也是个好选择。
输出：
Spiritual_and_religious_value	1	玩的很开心
Knowledge_and_education	1	自然科普
Aesthetic_values_nature	1	小溪
Social_relations	1	带孩子
Recreation_and_visitation	1	爬山
PRIMARY	Social_relations
CONF	1.0

示例 11
评论：
暑假带着孩子家人去了一次颐和园，十七孔桥壮观雄伟。站在桥上看风景拍照非常不错，非常值得一来。以后有机会还会和家人朋友再去逛逛。
输出：
Inspiration	1	看风景拍照
Aesthetic_values_architecture	1	十七孔桥壮观雄伟
Aesthetic_object_landscape	1	看风景
Social_relations	1	家人朋友
Recreation_and_visitation	1	逛逛
PRIMARY	Recreation_and_visitation
CONF	1.0
"""

SYSTEM = (
    "你是中文游客评论的 CES 标注员。\n"
    f"{CONTEXT_KNOWLEDGE}\n"
    "请注意：在对'Spiritual_and_religious_value'类别进行标注时，要根据语境判断，避免将'老佛爷'、'佛香阁'等作为特有名词出现的词语误判为宗教或精神价值。\n"
    "只输出纯文本，不要 JSON，不要解释。\n"
    "输出格式（每行一条）：\n"
    "label\tsentiment\tevidence\n"
    "sentiment: 1 为正向，0 为负向/中性。\n"
    "最后两行必须是：\n"
    "PRIMARY\t<一个label或空>\n"
    "CONF\t<0到1的小数>\n"
    "label 只能从以下列表选：\n"
    f"{CES_CLASSES}\n"
    "evidence 不超过80字，必须来自评论，且不要包含制表符和换行。\n"
    "如果没有任何 CES：不要输出任何 label 行，PRIMARY 为空，CONF=0。\n"
    f"{FEW_SHOT_SECTION}"
)

def build_user_prompt(text: str, hints: list):
    return (
        "请根据背景知识并模仿上述示例，对以下评论进行标注：\n"
        f"评论：{text}\n\n"
        f"词库提示标签（优先考虑）：{hints}"
    )

# =========================
# 7. Utility functions
# =========================
def _throttle():
    global _last_call
    now = time.time()
    wait = MIN_INTERVAL_SEC - (now - _last_call)
    if wait > 0:
        time.sleep(wait)
    _last_call = time.time()

def make_comment_id(text: str):
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

def guess_text_col(df: pd.DataFrame):
    candidates = ["评论内容", "comment_detail", "content", "评论", "comment", "review", "text", "内容", "评价"]
    for c in candidates:
        if c in df.columns:
            if df[c].astype(str).str.strip().str.len().sum() > 0:
                return c
    obj_cols = [c for c in df.columns if df[c].dtype == "object" and "图片" not in c and "image" not in c.lower()]
    if not obj_cols: return None
    lens = {c: df[c].astype(str).map(len).mean() for c in obj_cols if df[c].astype(str).str.strip().str.len().sum() > 0}
    if lens: return max(lens, key=lens.get)
    return None

def read_csv_robust(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv(path, encoding="gbk")

# =========================
# 8. SQLite cache
# =========================
def init_db(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA busy_timeout=8000;")
    conn.execute("""
    CREATE TABLE IF NOT EXISTS responses (
      comment_id TEXT PRIMARY KEY,
      status TEXT,
      response_json TEXT,
      updated_at TEXT
    )
    """)
    conn.commit()
    return conn

def cache_set(conn, cid: str, status: str, obj):
    cur = conn.cursor()
    cur.execute("""
      INSERT INTO responses(comment_id,status,response_json,updated_at)
      VALUES(?,?,?,?)
      ON CONFLICT(comment_id) DO UPDATE SET
        status=excluded.status,
        response_json=excluded.response_json,
        updated_at=excluded.updated_at
    """, (cid, status, json.dumps(obj, ensure_ascii=False), datetime.now().isoformat(timespec="seconds")))
    conn.commit()
    cur.close()

# =========================
# 9. Core google-genai API call with enhanced parsing, lexicon fallback, timeouts, and retries with backoff
# =========================
def parse_tsv_output(raw: str):
    raw = (raw or "").strip()
    raw = raw.replace('```tsv', '').replace('```', '').strip()
    lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
    label_evidence = []
    primary, conf = "", 0.0
    for ln in lines:
        parts = re.split(r'\t+', ln)
        if len(parts) < 2:
            parts = re.split(r'\s{2,}', ln)
            if len(parts) < 2: continue
        k = parts[0].strip().upper()
        if k == "PRIMARY": primary = parts[1].strip()
        elif k == "CONF":
            try:
                match = re.search(r"[0-9.]+", parts[1])
                conf = float(match.group()) if match else 0.0
            except: conf = 0.0
        elif parts[0].strip() in CES_CLASSES:
            label = parts[0].strip()
            sentiment = 1
            evidence = ""
            if len(parts) >= 3:
                try: sentiment = int(re.search(r"\d", parts[1]).group())
                except: sentiment = 1
                evidence = parts[2].strip()
            elif len(parts) == 2:
                evidence = parts[1].strip()
            label_evidence.append({"label": label, "sentiment": sentiment, "evidence": evidence[:80]})
    seen = set()
    cleaned = [item for item in label_evidence if not (item["label"] in seen or seen.add(item["label"])) ]
    return {"label_evidence": cleaned, "primary_label": primary, "confidence": conf}

def call_api(text: str, hints: list) -> dict:
    prompt_text = f"{SYSTEM}\n\n{build_user_prompt(text, hints)}"

    # Set the timeout in http_options in milliseconds.
    config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=800,
        http_options={'timeout': 30000} # Require a response or error within 30 seconds.
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            _throttle() # Respect MIN_INTERVAL_SEC before each attempt

            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt_text,
                config=config
            )
            raw = response.text
            if not raw: raise RuntimeError("Empty Response")

            obj = parse_tsv_output(raw)

            # --- Lexicon fallback logic ---
            if hints and (not obj["label_evidence"] or obj["confidence"] < MIN_CONFIDENCE_FOR_FALLBACK):
                fallback_items = []
                for h in hints:
                    if h in CES_CLASSES:
                        fallback_items.append({"label": h, "sentiment": 1, "evidence": evidence_from_text(h, text)})
                if fallback_items:
                    obj["label_evidence"] = fallback_items
                    obj["confidence"] = max(obj["confidence"], 0.35)
            return obj

        except Exception as e:
            error_str = str(e)
            if "429" in error_str:
                # Exponential backoff for 429 errors
                wait_time = (2 ** (attempt - 1)) * 5 + random.uniform(0, 1)
                print(f"⚠️ 触发限速 (429)，正在进行第 {attempt} 次重试，等待 {wait_time:.1f} 秒...")
                time.sleep(wait_time)
            else:
                # Fixed sleep for other errors (e.g., timeouts, 500, network issues)
                print(f"❌ Attempt {attempt} failed: {e}，休息 2 秒...")
                time.sleep(2)
            if attempt == MAX_RETRIES:
                print(f"所有 {MAX_RETRIES} 次重试均失败。")
                return {"label_evidence": [], "primary_label": "", "confidence": 0.0, "_error": "ALL_ATTEMPTS_FAILED"}

# =========================
# 10. Main workflow
# =========================
INPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/源数据-重分类/baidumap/香山-植物园2")
OUTPUT_BASE_DIR = Path(r"/content/drive/MyDrive/CES分析/评论-分析结果/香山-植物园2")

if not INPUT_BASE_DIR.exists():
    raise FileNotFoundError(f"找不到输入文件夹：{INPUT_BASE_DIR}")

OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"输出文件夹已准备好：{OUTPUT_BASE_DIR}")

db_path = OUTPUT_BASE_DIR / "_ces_cache.sqlite"
conn = None

def flush(rows: list, current_detail_csv: Path, write_header: bool):
    if not rows: return
    part = pd.DataFrame(rows)
    base_cols = ["original_index", "name", "user_name", "comment_id", "text", "CES_labels", "CES_confidence"]
    all_cols = base_cols + CES_DATA_COLS + ["status", "updated_at"]

    for c in all_cols:
        if c not in part.columns: part[c] = ""

    part = part[all_cols]
    current_detail_csv.parent.mkdir(parents=True, exist_ok=True)
    part.to_csv(current_detail_csv, mode="a", index=False, header=write_header, encoding="utf-8-sig")

def get_timestamp_from_filename(filepath: Path) -> datetime:
    match = re.search(r'-(\d{8}_\d{4})\.csv$', filepath.name)
    if match:
        return datetime.strptime(match.group(1), '%Y%m%d_%H%M')
    return datetime.min

all_input_csv_files = list(INPUT_BASE_DIR.glob('**/*.csv'))
if not all_input_csv_files:
    print(f"在 {INPUT_BASE_DIR} 及其子文件夹中未找到任何 CSV 文件。")
else:
    print(f"找到 {len(all_input_csv_files)} 个 CSV 文件进行处理。")

total_processed_files = 0

try:
    conn = init_db(db_path)

    for file_idx, input_file_path in enumerate(all_input_csv_files):
        print(f"\n--- 处理文件 {file_idx+1}/{len(all_input_csv_files)}: {input_file_path.name} ---")
        sub_attraction_name = input_file_path.stem if input_file_path.parent == INPUT_BASE_DIR else input_file_path.parent.name
        current_detail_csv = None
        write_header = True

        try:
            df = read_csv_robust(input_file_path)
            text_col = guess_text_col(df)
            if text_col is None:
                print(f"警告: 文件 {input_file_path.name} 中未找到合适的文本列，跳过。")
                continue

            expected_cids_from_input = set()
            for text_content in df[text_col].dropna().astype(str):
                if text_content.strip():
                    expected_cids_from_input.add(make_comment_id(text_content.strip()))

            matching_output_files = []
            for existing_output_candidate in OUTPUT_BASE_DIR.glob('*.csv'):
                match_output_name = re.match(r"^(.*?)-(\d{8}_\d{4})\.csv$", existing_output_candidate.name)
                if match_output_name and match_output_name.group(1) == sub_attraction_name:
                    matching_output_files.append(existing_output_candidate)

            if matching_output_files:
                latest_output_csv = max(matching_output_files, key=get_timestamp_from_filename)
                try:
                    existing_output_df = read_csv_robust(latest_output_csv)
                    if "comment_id" in existing_output_df.columns:
                        actual_cids_in_output = set(existing_output_df["comment_id"].dropna().astype(str).tolist())
                    else:
                        actual_cids_in_output = set()

                    if expected_cids_from_input.issubset(actual_cids_in_output) and len(actual_cids_in_output) >= len(expected_cids_from_input):
                        print(f"文件 {latest_output_csv.name} 已完整处理，跳过。")
                        total_processed_files += 1
                        continue
                    else:
                        current_detail_csv = latest_output_csv
                        write_header = False
                        print(f"检测到未完成的输出文件: {latest_output_csv.name}，将继续处理。")
                except Exception as e:
                    print(f"警告: 读取现有输出文件 {latest_output_csv.name} 失败 ({e})，将创建新文件。")

            if current_detail_csv is None:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M")
                current_detail_csv = OUTPUT_BASE_DIR / f"{sub_attraction_name}-{timestamp}.csv"
                print(f"未检测到有效输出文件: {current_detail_csv.name}")
                actual_cids_in_output = set()

            buffer = []
            processed_items_count = 0
            total_comments = len(df)

            for idx, row_data in df.iterrows():
                text = str(row_data.get(text_col, "")).strip()
                if not text: continue

                cid = make_comment_id(text)
                if cid in actual_cids_in_output:
                    processed_items_count += 1
                    continue

                hints = lexicon_hint_labels(text)
                print(f"[{file_idx+1}/{len(all_input_csv_files)}] 文件 [{input_file_path.name}] - 正在处理评论 {processed_items_count+1}/{total_comments}...")
                processed_items_count += 1

                res = call_api(text, hints)
                le = res.get("label_evidence", [])
                label_info = {item["label"]: item for item in le if "label" in item}

                output_row = {
                    "original_index": idx,
                    "name": row_data.get("name", ""),
                    "user_name": row_data.get("user_name", ""),
                    "comment_id": cid,
                    "text": text[:TEXT_MAX_CHARS],
                    "CES_labels": ";".join(label_info.keys()),
                    "CES_confidence": res.get("confidence", 0.0),
                    "status": "ok",
                    "updated_at": datetime.now().isoformat(timespec="seconds")
                }

                for c in CES_CLASSES:
                    if c in label_info:
                        output_row[f"EV_{c}"] = label_info[c].get("evidence", "")
                        output_row[f"SNT_{c}"] = label_info[c].get("sentiment", 1)
                    else:
                        output_row[f"EV_{c}"] = ""
                        output_row[f"SNT_{c}"] = ""

                buffer.append(output_row)
                cache_set(conn, cid, "ok", res)

                # Add a 2-second pause to avoid filling the rolling window too quickly.
                print(f"😴 强制休息 2 秒...")
                time.sleep(2)

                if len(buffer) >= WRITE_EVERY:
                    flush(buffer, current_detail_csv, write_header)
                    buffer = []
                    write_header = False

            if buffer:
                flush(buffer, current_detail_csv, write_header)
            print(f"文件 {input_file_path.name} 处理完成！结果保存至: {current_detail_csv}")
            total_processed_files += 1

        except Exception as e:
            print(f"处理文件 {input_file_path.name} 时发生错误: {e}")

finally:
    if conn:
        try:
            conn.close()
            time.sleep(0.1)
        except Exception as close_e:
            print(f"Error closing DB: {close_e}")

print(f"\n所有文件处理完毕！共处理 {total_processed_files} 个文件。")


### Check that all files were processed